# Calculate the ESPN score

***

### Input what year we are doing and what ranking method and weights do we want? 

In [15]:
import pandas as pd
import numpy as np 
import datetime

gameFilename = '2015AllGames.txt'
gameFilename2 = '2015Games.txt'
teamFilename = '2015Teams.txt'

# Set weights for home, away and neutral wins
weightHomeWin = 1
weightAwayWin = 1
weightNeutralWin = 1
segmentWeighting = [1, 1]

# Will you use weighting? 
useWeighting = True

### Function to convert a date to the Massey date that appears in the first column of the data.

In [16]:
def convertDateToMassey(dateToConvert): 
    # Note, there are 718528 days between January 1, 1970 and (what would be) January 1, year 0. 

    return (dateToConvert - datetime.datetime(1970,1,1)).days + 719528

### Load the file names and then the teams and games

In [17]:
teamNames = pd.read_csv(teamFilename, header = None)
teamNames.loc[:,0] = teamNames.loc[:,0] - 1
numTeams = len(teamNames)

def colleyRanking():
    ##### columns of games are:
    #	column 0 = days since 1/1/0000
    #	column 1 = date in YYYYMMDD format
    
    #	column 2 = team1 index
    #	column 3 = team1 homefield (1 = home, -1 = away, 0 = neutral)
    #	column 4 = team1 score
    #	column 5 = team2 index
    #	column 6 = team2 homefield (1 = home, -1 = away, 0 = neutral)
    #	column 7 = team2 score
    games = pd.read_csv(gameFilename2, header = None)
    numGames = len(games)
    # print(games[2][5265])
    # print (weight_avg_spl(71))
    # print (weight_avg_spl(76))
     # if team2ID in avg_spl:
     #            gameweight=weight_avg_spl(team2ID)*timeWeight

    import numpy as np
    from math import ceil
    
    colleyMatrix = 2*np.diag(np.ones(numTeams))
    b = np.ones(numTeams)
    
    dayBeforeSeason = games.loc[0,0] - 1
    lastDayOfSeason = games.loc[len(games)-1,0]
    
    for i in range(numGames):
        team1ID = games.loc[i, 2] - 1 # subtracting 1 since python indexes at 0
        team1Score = games.loc[i, 4]
        team1Loc = games.loc[i, 3];
    
        team2ID = games.loc[i, 5] - 1 # subtracting 1 since python indexes at 0
        team2Score = games.loc[i, 7]
        team2Loc = games.loc[i, 6];
        
        currentDay = games.loc[i,0]
    
        # Find the weight for this game using time and home/away    
        if useWeighting:
            numberSegments = len(segmentWeighting)
            weightIndex = ceil(numberSegments*((currentDay-dayBeforeSeason)/(lastDayOfSeason-dayBeforeSeason))) - 1
            timeWeight = segmentWeighting[weightIndex]
        else:
            timeWeight = 1
    
        if team1Score > team2Score:  # Team 1 won        
            if (team1Loc == 1):      # Home win
                gameWeight = weightHomeWin*timeWeight
            elif (team1Loc == -1):   # Away win
                gameWeight = weightAwayWin*timeWeight
            else:                    # Neutral court win
                gameWeight = weightNeutralWin*timeWeight
        else:                        # Team 2 won
            if (team2Loc == 1):      # Home win
                gameWeight = weightHomeWin*timeWeight
            elif (team2Loc == -1):   # Away win
                gameWeight = weightAwayWin*timeWeight
            else:                    # Neutral court win
                gameWeight = weightNeutralWin*timeWeight
    
            
                    
        # Update the Colley matrix and RHS
        colleyMatrix[team1ID, team2ID] -= gameWeight
        colleyMatrix[team2ID, team1ID] -= gameWeight
    
        colleyMatrix[team1ID, team1ID] += gameWeight
        colleyMatrix[team2ID, team2ID] += gameWeight
        
        if team1Score > team2Score:
            b[team1ID] += 1/2*gameWeight
            b[team2ID] -= 1/2*gameWeight
        elif team1Score < team2Score:
            b[team1ID] -= 1/2*gameWeight
            b[team2ID] += 1/2*gameWeight
        else:  # it is a tie and make 1/2 a win and 1/2 a loss for both teams
            b[team1ID] += 0; # this equates to adding nothing
            b[team2ID] += 0; # clearly this code could be deleted

    ratings = np.linalg.solve(colleyMatrix,b)

    return(ratings)
    

# columns of games are:
#	column 0 = days since 1/1/0000
#	column 1 = date in YYYYMMDD format
#	column 2 = team1 index
#	column 3 = team1 homefield (1 = home, -1 = away, 0 = neutral)
#	column 4 = team1 score
#	column 5 = team2 index
#	column 6 = team2 homefield (1 = home, -1 = away, 0 = neutral)
#	column 7 = team2 score
games = pd.read_csv(gameFilename, header = None)
games.loc[:,2] = games.loc[:,2] - 1
games.loc[:,5] = games.loc[:,5] - 1
numGames = len(games)

selectionSundayList = ['03/10/2002','03/16/2003','03/14/2004','03/13/2005','03/12/2006',\
    '03/11/2007','03/16/2008','03/15/2009','03/14/2010','03/13/2011','03/11/2012',\
    '03/17/2013','03/16/2014','03/15/2015','03/13/2016','03/12/2017','03/11/2018', '3/17/2019','3/15/2020','3/14/2021']
 
yearOfTournament = int(str(games.iloc[-1,1])[0:4])
selectionSundayDate = datetime.datetime.strptime(selectionSundayList[yearOfTournament - 2002], "%m/%d/%Y")
selectionSundayInt = convertDateToMassey(selectionSundayDate)

numberGamesBeforeMadness = len(np.where(games[0] <= selectionSundayInt)[0])

### Find March Madness teams

In [18]:
marchMadnessTeams = {games.iloc[-1,2], games.iloc[-1,5]}

if games.iloc[-1,4] > games.iloc[-1,7]:
    nationalChampion = games.iloc[-1,2]
else:     
    nationalChampion = games.iloc[-1,5]
    
finalTwo = marchMadnessTeams
    
for i in reversed(range(numGames)): 
    teamsInGame = {games.iloc[i,2], games.iloc[i,5]}
    if len(teamsInGame & marchMadnessTeams) > 0:        
        marchMadnessTeams = marchMadnessTeams.union(teamsInGame)

    if len(marchMadnessTeams) == 4:
        finalFour = marchMadnessTeams
    elif len(marchMadnessTeams) == 8:
        eliteEight = marchMadnessTeams
    elif len(marchMadnessTeams) == 16:
        sweet16 = marchMadnessTeams
    elif len(marchMadnessTeams) == 32:
        round32 = marchMadnessTeams

    if len(marchMadnessTeams) >= 64:
        indexToStart = i;
        break

marchMadnessTeamsByRoundList = []
marchMadnessTeamsByRoundList.append(list(marchMadnessTeams))
marchMadnessTeamsByRoundList.append(list(round32))
marchMadnessTeamsByRoundList.append(list(sweet16))
marchMadnessTeamsByRoundList.append(list(eliteEight))
marchMadnessTeamsByRoundList.append(list(finalFour))
marchMadnessTeamsByRoundList.append(list(finalTwo))
marchMadnessTeamsByRoundList.append([nationalChampion])
print(marchMadnessTeamsByRoundList)

[[131, 259, 262, 135, 9, 137, 13, 273, 147, 20, 19, 151, 281, 158, 31, 32, 289, 167, 169, 299, 300, 45, 302, 49, 306, 184, 316, 190, 64, 65, 321, 323, 196, 325, 326, 73, 202, 78, 206, 208, 337, 211, 340, 214, 343, 215, 216, 345, 219, 348, 347, 121, 98, 99, 100, 102, 232, 105, 233, 107, 240, 249, 124, 125], [65, 131, 325, 326, 135, 73, 9, 202, 13, 208, 337, 147, 211, 340, 214, 343, 215, 219, 348, 158, 32, 98, 100, 102, 167, 300, 45, 124, 306, 249, 316, 190], [102, 167, 135, 73, 9, 202, 337, 306, 147, 211, 340, 348, 343, 215, 316, 190], [102, 167, 135, 73, 9, 147, 211, 343], [167, 135, 73, 343], [73, 343], [73]]


### To find predicted bracket, we first rank! 

In [19]:
r = colleyRanking()

### Now, find the round-by-round match-ups in your bracket and predicted winners.

In [20]:
marchMadnessTeamsList = list(marchMadnessTeams)
marchMadnessCorrect = np.zeros(len(marchMadnessTeams))

# Create initial list of pairings as teams by themselves.  
initList = []
for i in range(len(marchMadnessTeamsList)):
     initList.append([marchMadnessTeamsList[i]])
pairingsList = [initList]

# We start with no correct predictions! 
madnessCorrect = np.zeros(len(marchMadnessTeams))

# Skip play-in games 
thursdayOfMadness = selectionSundayInt + 4
numberGamesBeforeFirstRound = len(np.where(games[0] < thursdayOfMadness)[0])

currentRoundList = []
currentRound = 0
teamsInRound = [64,32,16,8,4,2]
gamesInRound = 0

# First round contains every team 
roundByRoundTeamsList =[]
currentRoundByRoundTeamsList = []

currentPairingsList = pairingsList[-1]
#print('ROUND %d' % (currentRound + 1))
for i in range(numberGamesBeforeFirstRound,numGames): 
    team1ID = games.loc[i, 2] 
    team1Score = games.loc[i, 4]
    team2ID = games.loc[i, 5] 
    team2Score = games.loc[i, 7]    
    
    # Are March Madness teams in the game? Then, it's a Madness game
    if len({team1ID,team2ID} & marchMadnessTeams) == 2:
        # Figure out who would play in your bracket! 
        gamesInRound += 1 
        
        for k in range(len(currentPairingsList)):
            if team1ID in currentPairingsList[k]:
                team1Teams = np.array(currentPairingsList[k])
            elif team2ID in currentPairingsList[k]:
                team2Teams = np.array(currentPairingsList[k])
                
        # Find teams predicted to play, which have maximum rating 
        # in the list of teams who have played to this point
        predictedTeam1ID = team1Teams[np.argmax(r[team1Teams])]
        predictedTeam2ID = team2Teams[np.argmax(r[team2Teams])]
        currentRoundByRoundTeamsList.append(predictedTeam1ID)
        currentRoundByRoundTeamsList.append(predictedTeam2ID)
        
        currentRoundList.append(list(team1Teams) + list(team2Teams))
        
        if r[predictedTeam1ID] > r[predictedTeam2ID]:
            predictedWinner = predictedTeam1ID
        else: 
            predictedWinner = predictedTeam2ID
               
        if team1Score > team2Score:
            actualWinner = team1ID
        else: 
            actualWinner = team2ID
            
#        print('   %s (%d) vs. %s (%d) >>>> %s' % (teamNames.loc[predictedTeam1ID,1],predictedTeam1ID, teamNames.loc[predictedTeam2ID,1],predictedTeam2ID,teamNames.loc[predictedWinner,1]))
        
        if actualWinner == predictedWinner: 
            indexOfPick = marchMadnessTeamsList.index(actualWinner)
            madnessCorrect[indexOfPick] += 1
#            print('       CORRECT PREDICTION')
        
        if (gamesInRound == teamsInRound[currentRound]/2):
            currentRound += 1
            pairingsList.append(currentRoundList)
            currentRoundList = []
            gamesInRound = 0
            currentPairingsList = pairingsList[-1]
            roundByRoundTeamsList.append(currentRoundByRoundTeamsList)
            currentRoundByRoundTeamsList = []
#            if currentRound < 6:
#                print('ROUND %d' % (currentRound + 1))

espnScore = 0
espnPoints = [0,10,30,70,150,310,630]
for i in range(len(espnPoints)): 
    espnScore += espnPoints[i]*len(np.where(madnessCorrect == i)[0])                
    
print('\n\n>>>>>>>>>>>>>>>>>>>>>>>>>>')
print('ESPN Score = %d' % espnScore) 
print('>>>>>>>>>>>>>>>>>>>>>>>>>>')

print("Correct predictions per team:", madnessCorrect)
print("Number of picks with each score tier:")
for i in range(len(espnPoints)): 
    print(f"{i} correct picks: {len(np.where(madnessCorrect == i)[0])}")

    



>>>>>>>>>>>>>>>>>>>>>>>>>>
ESPN Score = 840
>>>>>>>>>>>>>>>>>>>>>>>>>>
Correct predictions per team: [1. 0. 0. 4. 3. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 0. 0. 1. 0. 0.
 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 4. 2. 0. 0. 1. 1. 2. 1. 0. 4. 2. 0. 0.
 1. 1. 0. 0. 1. 0. 0. 3. 0. 0. 0. 0. 0. 0. 0. 0.]
Number of picks with each score tier:
0 correct picks: 40
1 correct picks: 16
2 correct picks: 3
3 correct picks: 2
4 correct picks: 3
5 correct picks: 0
6 correct picks: 0


### Print Madness ratings

In [21]:
################################################
###### PRINT THE MARCH MADNESS TEAMS AND RANKING
################################################
iSort = np.argsort(-r)

print('===========================')
print('Rank   Rating    Team   ')
print('===========================')

iRank = 0
for i in range(numTeams):
    if iSort[i] in marchMadnessTeams: 
        print(f'{iRank+1:4d}   {r[iSort[i]]:.5f}  {teamNames.loc[iSort[i],1]}')
        iRank += 1

print('')   # extra carriage returmadnessCorrect

Rank   Rating    Team   
   1   1.13344   Kentucky
   2   1.08453   Villanova
   3   1.06458   Wisconsin
   4   1.04553   Virginia
   5   1.04104   Duke
   6   1.03463   Arizona
   7   1.00896   Kansas
   8   0.99849   Gonzaga
   9   0.95157   Notre_Dame
  10   0.94454   Iowa_St
  11   0.93718   Maryland
  12   0.91509   Baylor
  13   0.90815   Northern_Iowa
  14   0.90426   North_Carolina
  15   0.90307   Louisville
  16   0.89933   SMU
  17   0.89592   Wichita_St
  18   0.89179   Arkansas
  19   0.88620   Oklahoma
  20   0.88320   West_Virginia
  21   0.87616   VCU
  22   0.86888   Georgetown
  23   0.86606   Utah
  24   0.86453   Providence
  25   0.85668   Oregon
  26   0.85377   Michigan_St
  27   0.85068   Butler
  28   0.82084   Dayton
  29   0.81943   St_John's
  30   0.81639   San_Diego_St
  31   0.81608   Davidson
  32   0.81205   Xavier
  33   0.80977   Ohio_St
  34   0.80575   Georgia
  35   0.79646   Iowa
  36   0.79183   Texas
  37   0.79131   LSU
  38   0.79017   NC_Stat

In [22]:
print('\n****************************')
print('   ACTUAL MADNESS RESULTS')
print(  '****************************')
print('\nNational Champion was %s' % teamNames.loc[nationalChampion,1])

print('\nChampionship Game')
print('=================')
for i in finalTwo:
    print('%3d %s' % (i,teamNames.loc[i,1]))
        
print('\nFinal Four')
print('==========')
for i in finalFour:
    print('%3d %s' % (i,teamNames.loc[i,1]))

print('\nElite 8')
print('==========')
for i in eliteEight:
    print('%3d %s' % (i,teamNames.loc[i,1]))

print('\nSweet 16')
print('==========')
for i in sweet16:
    print('%3d %s' % (i,teamNames.loc[i,1]))

print('\nRound of 32')
print('==========')
for i in round32:
    print('%3d %s' % (i,teamNames.loc[i,1]))
    
#print('\nMadness python team indices')
#print(marchMadnessTeams)


****************************
   ACTUAL MADNESS RESULTS
****************************

National Champion was  Duke

Championship Game
 73  Duke
343  Wisconsin

Final Four
167  Michigan_St
135  Kentucky
 73  Duke
343  Wisconsin

Elite 8
102  Gonzaga
167  Michigan_St
135  Kentucky
 73  Duke
  9  Arizona
147  Louisville
211  Notre_Dame
343  Wisconsin

Sweet 16
102  Gonzaga
167  Michigan_St
135  Kentucky
 73  Duke
  9  Arizona
202  North_Carolina
337  West_Virginia
306  UCLA
147  Louisville
211  Notre_Dame
340  Wichita_St
348  Xavier
343  Wisconsin
215  Oklahoma
316  Utah
190  NC_State

Round of 32
 65  Dayton
131  Kansas
325  Villanova
326  Virginia
135  Kentucky
 73  Duke
  9  Arizona
202  North_Carolina
 13  Arkansas
208  Northern_Iowa
337  West_Virginia
147  Louisville
211  Notre_Dame
340  Wichita_St
214  Ohio_St
343  Wisconsin
215  Oklahoma
219  Oregon
348  Xavier
158  Maryland
 32  Butler
 98  Georgetown
100  Georgia_St
102  Gonzaga
167  Michigan_St
300  UAB
 45  Cincinnati
124  Iowa


In [23]:
print('\n*******************************')
print('   PREDICTED MADNESS RESULTS')
print(  '*******************************')
print('\nNational Champion: %s' % teamNames.loc[iSort[0],1])

print('\nChampionship Game')
print('=================')
for i in roundByRoundTeamsList[5]:
    print('%3d %s' % (i,teamNames.loc[i,1]))
        
print('\nFinal Four')
print('==========')
for i in roundByRoundTeamsList[4]:
    print('%3d %s' % (i,teamNames.loc[i,1]))

print('\nElite 8')
print('==========')
for i in roundByRoundTeamsList[3]:
    print('%3d %s' % (i,teamNames.loc[i,1]))

print('\nSweet 16')
print('==========')
for i in roundByRoundTeamsList[2]:
    print('%3d %s' % (i,teamNames.loc[i,1]))

print('\nRound of 32')
print('==========')
for i in roundByRoundTeamsList[1]:
    print('%3d %s' % (i,teamNames.loc[i,1]))
    
#print('\nMadness python team indices')
#print(marchMadnessTeams)


*******************************
   PREDICTED MADNESS RESULTS
*******************************

National Champion:  Kentucky

Championship Game
325  Villanova
135  Kentucky

Final Four
 73  Duke
325  Villanova
343  Wisconsin
135  Kentucky

Elite 8
343  Wisconsin
  9  Arizona
135  Kentucky
131  Kansas
326  Virginia
325  Villanova
 73  Duke
102  Gonzaga

Sweet 16
  9  Arizona
 19  Baylor
211  Notre_Dame
131  Kansas
343  Wisconsin
202  North_Carolina
135  Kentucky
158  Maryland
102  Gonzaga
125  Iowa_St
208  Northern_Iowa
325  Villanova
 73  Duke
 98  Georgetown
326  Virginia
215  Oklahoma

Round of 32
262  SMU
125  Iowa_St
135  Kentucky
 45  Cincinnati
  9  Arizona
323  VCU
348  Xavier
 19  Baylor
151  LSU
325  Villanova
316  Utah
 98  Georgetown
202  North_Carolina
 13  Arkansas
211  Notre_Dame
 32  Butler
167  Michigan_St
326  Virginia
 73  Duke
273  St_John's
340  Wichita_St
131  Kansas
215  Oklahoma
232  Providence
102  Gonzaga
 64  Davidson
343  Wisconsin
219  Oregon
337  West_Virgin